# ZelusBench: Selective Attention (Long) — Noise 4.0-5.0x

Can the model filter heavy distractor noise around the critical chain?

**Independent variable**: noise multiplier ∈ [4.0, 5.0]
- Chain depth fixed at 5 → num_points = [20, 25]
- At 4.0x, 75% are distractors; at 5.0x, 80% are distractors
- POSITION queries only, targeting deep points
- transform_prob=0.1, dim=3 fixed
- 10 seeds per noise level = 20 scenarios

In [ ]:
import kaggle_benchmarks as kbench
import numpy as np
import random
import time

from zelusbench.scenarios.config import ScenarioConfig, QueryType
from zelusbench.scenarios.generator import ScenarioGenerator
from zelusbench.evaluation.scorer import score_query, score_response

DEPTH = 5
NOISE_LEVELS = [4.0, 5.0]
SEEDS = 10
TOTAL = len(NOISE_LEVELS) * SEEDS
print(f"ZelusBench Selective Attention (Long) — Noise 4.0-5.0x")
print(f"Depth: {DEPTH} | Noise levels: {NOISE_LEVELS} | Seeds: {SEEDS} | Total: {TOTAL} scenarios")

In [ ]:
def _prompt_with_retry(llm, prompt, attempts=4):
    for attempt in range(attempts):
        try:
            return llm.prompt(prompt)
        except Exception as e:
            if attempt == attempts - 1:
                raise
            wait = 2 ** attempt
            print(f"    !! API error ({type(e).__name__}), retry {attempt+1}/{attempts-1} in {wait}s")
            time.sleep(wait)

@kbench.task(name="zelusbench_selective_long")
def zelusbench_selective_long(llm) -> float:
    _start = time.time()
    print(f"Running {TOTAL} scenarios...")
    print("=" * 60)
    all_scenario_avgs = []
    level_scores = {}
    scenario_num = 0
    for noise in NOISE_LEVELS:
        num_pts = round(DEPTH * noise)
        level_scores[noise] = []
        for i in range(SEEDS):
            scenario_num += 1
            bg_rng = random.Random(i * 7919)
            cfg = ScenarioConfig.randomize_except(bg_rng, pinned={
                "dim": 3,
                "min_chain_depth": DEPTH, "max_chain_depth": DEPTH,
                "num_points": num_pts,
                "transform_prob": 0.1,
                "query_types": [QueryType.POSITION],
                "query_min_depth": max(1, DEPTH - 2),
                "num_queries": 3,
                "seed": int(noise * 1000) + i,
            })
            scenario = ScenarioGenerator(cfg).generate(f"selective_long_{noise}_{i}")
            response = _prompt_with_retry(llm, scenario.prompt)
            scores = score_response(response, scenario)
            avg = float(np.mean([s.score for s in scores]))
            level_scores[noise].append(avg)
            all_scenario_avgs.append(avg)
            print(f"  [{scenario_num}/{TOTAL}] noise={noise}x pts={num_pts} avg={avg:.2f} | lb={cfg.leaf_bias}")
        print(f"  >> Noise {noise}x mean: {float(np.mean(level_scores[noise])):.3f}")
    print("\n" + "=" * 60)
    for noise in NOISE_LEVELS:
        num_pts = round(DEPTH * noise)
        avg = round(float(np.mean(level_scores[noise])), 3)
        print(f"  noise={noise}x  pts={num_pts:2d}  accuracy={avg:.3f}")
        kbench.assertions.assert_true(
            avg > 0.0,
            expectation=f"Selective long noise={noise}x ({num_pts} pts): accuracy={avg:.3f}"
        )
    overall = round(float(np.mean(all_scenario_avgs)), 3)
    print(f"\nScore: {overall:.3f} | {len(all_scenario_avgs)} scenarios | {time.time()-_start:.1f}s")
    return overall

zelusbench_selective_long.run(llm=kbench.llm)

In [ ]:
%choose zelusbench_selective_long